# Topic 4 — Support Vector Machines (SVM)

> **Goal of this notebook:** understand how an SVM finds the *maximum-margin* boundary between classes, how the **kernel trick** lets it handle non-linear data, and how the `C` and `gamma` knobs control it. We'll visualise decision boundaries and apply it to a real dataset.

**Contents**
1. Concept & intuition
2. The mathematics (margin, hinge loss, the kernel trick)
3. Key hyperparameters (C, gamma, kernels)
4. The data: 2D shapes + Breast Cancer
5. Visualising linear vs RBF decision boundaries
6. Training on a real dataset
7. Evaluation & hyperparameter tuning
8. Pros, cons & when to use
9. Summary

## 1. Concept & Intuition

Many lines can separate two classes — an SVM picks the one with the **widest margin**: the boundary that sits as far as possible from the nearest points of each class. Those nearest points are the **support vectors**, and they alone define the boundary.

A wide margin generalises better. When classes can't be split by a straight line, SVM uses the **kernel trick** to bend the boundary — implicitly mapping data into a higher-dimensional space where a linear split *does* exist.

## 2. The Mathematics

For a linear SVM with labels $y_i \in \{-1, +1\}$, we want the boundary $\mathbf{w}\cdot\mathbf{x} + b = 0$ that **maximises the margin** $\frac{2}{\lVert \mathbf{w} \rVert}$. Equivalently, minimise:

$$\frac{1}{2}\lVert \mathbf{w} \rVert^2 + C \sum_i \max(0,\; 1 - y_i(\mathbf{w}\cdot\mathbf{x}_i + b))$$

The second term is the **hinge loss** — it penalises points inside the margin or misclassified. **C** trades off margin width vs training errors (the soft margin).

**The kernel trick:** the optimisation only needs *dot products* between points, $\mathbf{x}_i \cdot \mathbf{x}_j$. Replace that dot product with a **kernel** $K(\mathbf{x}_i, \mathbf{x}_j)$ and you get a non-linear boundary without ever computing the high-dimensional mapping. The popular **RBF kernel**:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \exp(-\gamma \lVert \mathbf{x}_i - \mathbf{x}_j \rVert^2)$$

## 3. Key Hyperparameters

| Parameter | Effect |
|-----------|--------|
| **C** | Low = wider margin, more tolerant of errors (more regularisation). High = fits training data harder (risk of overfit). |
| **gamma** (RBF) | Low = smooth, far-reaching influence. High = each point's influence is local → wiggly boundary, can overfit. |
| **kernel** | `linear`, `poly`, `rbf` (default), `sigmoid`. |

**Always scale features** — SVMs are distance-based and sensitive to feature scale.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons, load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
print('Libraries loaded.')

## 4. The Data

Two datasets: a synthetic **two-moons** set (non-linearly separable, 2D — perfect for *visualising* boundaries) and the real **Breast Cancer** set for a performance example.

In [ ]:
Xm, ym = make_moons(n_samples=300, noise=0.25, random_state=42)
plt.scatter(Xm[:, 0], Xm[:, 1], c=ym, cmap='coolwarm', edgecolor='k', s=25)
plt.title('Two-moons: not linearly separable')
plt.show()

## 5. Visualising Linear vs RBF Boundaries

A linear kernel can only draw a straight line; the RBF kernel curves around the moons.

In [ ]:
def plot_boundary(model, X, y, ax, title):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k', s=20)
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, kernel in zip(axes, ['linear', 'rbf']):
    clf = make_pipeline(StandardScaler(), SVC(kernel=kernel, C=1.0, gamma='scale'))
    clf.fit(Xm, ym)
    plot_boundary(clf, Xm, ym, ax, f'{kernel} kernel  (acc={clf.score(Xm, ym):.2f})')
plt.show()

In [ ]:
# Effect of gamma on the RBF boundary: too high = overfit wiggles
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, g in zip(axes, [0.1, 1, 30]):
    clf = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1.0, gamma=g))
    clf.fit(Xm, ym)
    plot_boundary(clf, Xm, ym, ax, f'gamma={g}')
plt.show()

## 6. Training on a Real Dataset

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

svm = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1.0, gamma='scale'))
svm.fit(X_train, y_train)
print('Test accuracy:', round(svm.score(X_test, y_test), 4))

## 7. Evaluation & Hyperparameter Tuning

SVM performance hinges on `C` and `gamma`. We grid-search them with cross-validation.

In [ ]:
pipe = make_pipeline(StandardScaler(), SVC(kernel='rbf'))
param_grid = {'svc__C': [0.1, 1, 10, 100], 'svc__gamma': ['scale', 0.01, 0.001]}
grid = GridSearchCV(pipe, param_grid, cv=5, n_jobs=-1)
grid.fit(X_train, y_train)

print('Best params:', grid.best_params_)
print('Best CV accuracy:', round(grid.best_score_, 4))
y_pred = grid.predict(X_test)
print('\n', classification_report(y_test, y_pred, target_names=data.target_names))

## 8. Pros, Cons & When to Use

**Pros**
- Very effective in high-dimensional spaces (even when features > samples).
- The kernel trick handles complex non-linear boundaries.
- Only support vectors matter → memory efficient.

**Cons**
- Slow to train on large datasets (poor scaling with sample count).
- No native probability output (needs extra calibration).
- Sensitive to `C`/`gamma` and to feature scaling — tuning is essential.

**When to use**
- Small-to-medium datasets with clear margins, especially high-dimensional ones (text, genomics).
- When accuracy matters more than training speed or probability outputs.

## 9. Summary

- SVM finds the **maximum-margin** separator, defined by a few **support vectors**.
- The **kernel trick** (e.g. RBF) bends the boundary to fit non-linear data without explicit high-dimensional mapping.
- **C** controls the margin/error trade-off; **gamma** controls how local the RBF influence is — too high overfits, as the boundary plots showed.
- With tuned `C`/`gamma`, the RBF SVM classified breast-cancer tumours with high accuracy.